# Базовые проверки (лаб. 2)
Разностная проверка `hess_vec` (п. 1.4, п. 2.1) и короткий тест линейного CG на SPD-системе.


In [1]:
%load_ext autoreload
%autoreload 2


In [2]:
%matplotlib inline
import sys
from pathlib import Path

_root = Path.cwd().resolve()
if _root.name == "notebooks":
    _root = _root.parent
_src = _root / "src"
if str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

import numpy as np
from experiments_common import init_notebook
from ml_tools import sparse_oracle_ops
from oracles import SmoothedSVML2Oracle, hess_vec_finite_diff
from optimization import linear_conjugate_gradients

init_notebook()

rng = np.random.default_rng(0)
m, n = 12, 6
A = rng.standard_normal((m, n))
b = np.sign(rng.standard_normal(m))
mvAx, mvATx, mm = sparse_oracle_ops(A)
oracle = SmoothedSVML2Oracle(mvAx, mvATx, mm, b, 0.1)
x = rng.standard_normal(n)
v = rng.standard_normal(n)
hv = oracle.hess_vec(x, v)
fd = hess_vec_finite_diff(oracle.func, x, v)
err = np.linalg.norm(hv - fd) / (np.linalg.norm(hv) + 1e-12)
print("Относительная ошибка hess_vec vs FD:", err)
assert err < 0.05

n2 = 20
M = rng.standard_normal((n2, n2))
S = M.T @ M + n2 * np.eye(n2)
bb = rng.standard_normal(n2)
x0 = np.zeros(n2)
xv, msg, _ = linear_conjugate_gradients(lambda z: S.dot(z), bb, x0, tolerance=1e-10)
print("linear CG:", msg, "||Sx-b||", np.linalg.norm(S.dot(xv) - bb))
assert msg == "success"


Matplotlib is building the font cache; this may take a moment.


Относительная ошибка hess_vec vs FD: 2.3466451480433888e-05
linear CG: success ||Sx-b|| 1.8708004413680915e-10
